### LIME 패키지
- Local Interpretable Model-agnostic Explanations
- ML의 해석 가능성을 제공하는 패키지
- ML의 구조나 학습방식을 몰라도 결과에 대해서 해석할 수 있게 만든 패키지
-----
- 한계점 : 전체를 대변하는 게 아니라 Local 국소적, 지역적인 설명을 제공한다.
- 개별예측에 좀 더 포커스를 맞춰서 설명을 한다.
-----
- 다양한 모델을 사용해서 모델 의존하지 않고 적용 가능하다.
- -----
- Perturbation
    - 데이터 변형을 통해서 모델의 예측이 변하는 방식을 분석한다.
    - 예측값의 변화와 입력값의 특징(피처의 특징)간의 관게를 찾아낸다.
------
- 단순한 모델로 대체해서 설명을 쉽게 할 수 있다.
- 예를들어 복잡란 모델의 예측을 설명하기 어려우니깐, 복잡한 모델이 예측한 값을 가지고 단순한 선형회귀모델로 학습해서, 간단한 모델로 결과를 해석할 수 있다.
- 대체하는 모델은 설명하려는 예측에 근처 데이터에서 학습을 해서(전체 데이터셋을 다 확인하는 것이 아니라, 내가 궁금한 것 또는 특정 데이터 셋에서) 확인하여서 모델의 동작을 통해 해석을 진행한다.

### 작동원리
- 제공된 데이터와 y값을 살펴보고 선택
- 변형된 데이터를 생성
- 변형된 데이터셋을 모델에 다시 입력해서 예측값을 다시 출력할 수 있음
- 관찰된 예측값을 사용해서다시 대체모델에 학습을 하는 것
- 대체 모델의 결과를 기반으로 어떤 식으로 피처와 예측값에 영향을 주는지 가중치를 시각화해서 제공하게 된다.

### 패키지 메서드
- 주요 메서드
- lime.lime_tabular.LimeTabularExplainer:  국소적인 데이터 설명할 때 사용

- lime.lime_text.LimeTextExplainer: 텍스트 데이터를 분석할 때 단어나, 문장이 예측에 미친 영향들 볼 때 시각화

- lime.lime_image.LimeImageExplainer: 이미지 가지고 중요도 계산

- explain_instance : 특정 입력 데이터에 대한 설명을 생성하는 메서드

In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from lime import lime_tabular

# 가상의 신용 대출 데이터 생성
np.random.seed(42)
data = pd.DataFrame({
    "Income" : np.random.randint(2000, 10000, 1000), # 연소득
    "Age" : np.random.randint(20, 70, 1000),
    "LoanAmount" : np.random.randint(1000, 5000, 1000),
    "CreditScore" : np.random.randint(300, 850, 1000),
    "Approved" : np.random.choice([0,1], 1000) })

# 데이터 나누기
X = data[["Income", "Age", "LoanAmount", "CreditScore"]]
y = data["Approved"]
X_train, X_test, y_train,y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [7]:
# 일반 ML 모델을 통해 학습을 해보자
model = RandomForestClassifier(random_state = 42)
model.fit(X_train, y_train)

# 설명할 데이터 셋을 지정한다.
customer_data = X_test.iloc[0]

In [9]:
## 데이터를 변형한다.
## 여러 데이터 셋을 만들어서 추가하는 것
perturbed_data = resample(X_train, n_samples = 500, random_state=42, replace = True)

## 해당 데이터 셋에 변형을 줘보자!
perturbed_data['Income'] += np.random.randint(-1000, 1000, size=500)
perturbed_data['Age'] += np.random.randint(-5, 5, size=500)
perturbed_data['LoanAmount'] += np.random.randint(-500, 500, size=500)
perturbed_data['CreditScore'] += np.random.randint(-50, 50, size=500)

print(perturbed_data.head())

     Income  Age  LoanAmount  CreditScore
482    3572   34        3965          567
225    6714   50        4485          829
409    5079   61        3992          591
334    3221   49        3260          811
118    9661   20        4059          492
